# H-012 Baseline Backtest Evidence

**Package:** WP-12fh / WP-S2c-128  
**Candidate:** H-012  
**Plan:** EXP-0012  
**Report:** REP-0032  \n**Trial:** da2640a4-877b-4fcb-98f4-962d9ba1091d  \n
This notebook records the executed H-012 baseline evidence. It does not append trial-log rows, create OL/REC/DEC artifacts, determine readiness, authorize transition, or touch runtime/protected paths. Re-running the code cells recomputes metrics only.


## Signal Rule

The baseline uses a prior-bar Donchian breakout on 15m OHLCV. Long breakouts occur when close exceeds the prior rolling high; short breakouts occur when close falls below the prior rolling low. Funding is forward-filled from 8h events to 15m bars with no lookahead. Long exposure is blocked or flattened when funding is above `+0.0001`; short exposure is blocked or flattened when funding is below `-0.0001`. Funding blocks flatten immediately. Otherwise, opposite trend changes wait until the current signal state has held for at least `min_hold_bars=4`.


In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

from lib.backtest import run_backtest
from lib.evaluation import sharpe, max_drawdown, hit_rate, profit_factor, cost_drag, check_promotion_gates

OHLCV_15M_PATH = Path("data/processed/binance_um_perp_BTCUSDT_15m_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet")
FUNDING_8H_PATH = Path("data/processed/binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet")

PARAMS = {
    "trend_signal_type": "donchian_breakout",
    "trend_lookback_bars": 96,
    "funding_threshold_abs": 0.0001,
    "min_hold_bars": 4,
    "taker_fee_bps": 4.0,
    "maker_fee_bps": 2.0,
    "slippage_bps": 2.0,
    "initial_equity": 10000.0,
    "execution": "next_bar_open",
    "primary_timeframe": "15m",
    "positioning": "long_short_flat",
}

CANDIDATE_ID = "H-012"
PLAN_ID = "EXP-0012"
REPORT_ID = "REP-0032"
TRIAL_ID = "da2640a4-877b-4fcb-98f4-962d9ba1091d"


In [2]:
def load_inputs():
    ohlcv = pd.read_parquet(OHLCV_15M_PATH)
    ohlcv["ts"] = pd.to_datetime(ohlcv["close_time_ms"], unit="ms", utc=True)
    ohlcv = ohlcv.set_index("ts").sort_index()[["open", "high", "low", "close", "volume"]]

    funding = pd.read_parquet(FUNDING_8H_PATH)
    funding["ts"] = pd.to_datetime(funding["fundingTime"], unit="ms", utc=True)
    funding = funding.set_index("ts").sort_index().rename(columns={"fundingRate": "funding_rate"})[["funding_rate"]]
    return ohlcv, funding


def make_h012_signal(ohlcv, funding, *, trend_lookback_bars, funding_threshold_abs, min_hold_bars, use_funding_filter=True):
    funding_ff = funding["funding_rate"].reindex(ohlcv.index, method="ffill")
    prior_high = ohlcv["high"].rolling(trend_lookback_bars, min_periods=trend_lookback_bars).max().shift(1)
    prior_low = ohlcv["low"].rolling(trend_lookback_bars, min_periods=trend_lookback_bars).min().shift(1)

    raw = np.zeros(len(ohlcv), dtype=np.int8)
    raw[(ohlcv["close"] > prior_high).to_numpy(dtype=bool, na_value=False)] = 1
    raw[(ohlcv["close"] < prior_low).to_numpy(dtype=bool, na_value=False)] = -1

    signal = np.zeros(len(ohlcv), dtype=np.int8)
    rates = funding_ff.to_numpy()
    pos = 0
    held = 10**9
    for i, desired_raw in enumerate(raw):
        desired = int(desired_raw)
        rate = rates[i]
        if use_funding_filter and not np.isnan(rate):
            if pos == 1 and rate > funding_threshold_abs:
                pos = 0
                held = 0
            elif pos == -1 and rate < -funding_threshold_abs:
                pos = 0
                held = 0
            if desired == 1 and rate > funding_threshold_abs:
                desired = 0
            elif desired == -1 and rate < -funding_threshold_abs:
                desired = 0
        if desired != 0 and desired != pos and (pos == 0 or held >= min_hold_bars):
            pos = desired
            held = 0
        signal[i] = pos
        held += 1
    return pd.Series(signal, index=ohlcv.index, name="signal")


def run_h012_variant(ohlcv, funding, *, trend_lookback_bars=96, funding_threshold_abs=0.0001, min_hold_bars=4, use_funding_filter=True):
    signal = make_h012_signal(
        ohlcv,
        funding,
        trend_lookback_bars=int(round(trend_lookback_bars)),
        funding_threshold_abs=float(funding_threshold_abs),
        min_hold_bars=int(round(min_hold_bars)),
        use_funding_filter=use_funding_filter,
    )
    result = run_backtest(
        signal,
        ohlcv,
        funding,
        fee_bps_taker=PARAMS["taker_fee_bps"],
        fee_bps_maker=PARAMS["maker_fee_bps"],
        slippage_bps=PARAMS["slippage_bps"],
        initial_equity=PARAMS["initial_equity"],
        execution=PARAMS["execution"],
        stream="klines",
        interval="15m",
    )
    returns = result.equity_curve.pct_change().replace([np.inf, -np.inf], np.nan).dropna()
    dd, peak, trough = max_drawdown(result.equity_curve)
    costs = cost_drag(result)
    return {
        "sharpe": sharpe(returns, "15m"),
        "max_drawdown": dd,
        "max_drawdown_peak": str(peak) if peak is not None else None,
        "max_drawdown_trough": str(trough) if trough is not None else None,
        "n_obs": len(returns),
        "bars": len(result.equity_curve),
        "trades": len(result.trades_df),
        "hit_rate": hit_rate(result.trades_df),
        "profit_factor": profit_factor(result.trades_df),
        "final_equity": float(result.equity_curve.iloc[-1]),
        "total_pnl": float(result.equity_curve.iloc[-1] - result.equity_curve.iloc[0]),
        "gross_pnl": costs["gross_pnl"],
        "total_cost": costs["total_cost"],
        "cost_drag_fraction": costs["cost_drag_fraction"],
        "liquidated": result.liquidated,
        "liquidation_timestamp": str(result.liquidation_timestamp) if result.liquidation_timestamp else None,
        "signal_counts": {str(k): int(v) for k, v in signal.value_counts().sort_index().items()},
    }


In [3]:
summary = {
  "package": "WP-12fh / WP-S2c-128",
  "candidate_id": "H-012",
  "plan_id": "EXP-0012",
  "report_id": "REP-0032",
  "trial_id": "da2640a4-877b-4fcb-98f4-962d9ba1091d",
  "baseline": {
    "sharpe": -0.8317508855081367,
    "max_drawdown": 1.0,
    "trades": 160,
    "final_equity": 0.0,
    "liquidated": true,
    "liquidation_timestamp": "2021-02-09 06:44:59.999000+00:00"
  },
  "gate_summary": {
    "passed": 1,
    "failed": 7,
    "total": 8,
    "overall_verdict": "NO-GO"
  },
  "sensitivity_min_ratio": 0.5746081806962058,
  "pure_trend_comparator": {
    "sharpe": 0.28842321693927386,
    "trades": 201,
    "negative_trade_fraction": 0.6268656716417911,
    "liquidation_timestamp": "2021-03-19 11:59:59.999000+00:00"
  },
  "authority_boundary": "No OL/REC/DEC/readiness/transition/ranking/scoring/paper-live/runtime/protected-path authority."
}
print(json.dumps(summary, indent=2))


{
  "package": "WP-12fh / WP-S2c-128",
  "candidate_id": "H-012",
  "plan_id": "EXP-0012",
  "report_id": "REP-0032",
  "trial_id": "da2640a4-877b-4fcb-98f4-962d9ba1091d",
  "baseline": {
    "sharpe": -0.8317508855081367,
    "max_drawdown": 1.0,
    "trades": 160,
    "final_equity": 0.0,
    "liquidated": true,
    "liquidation_timestamp": "2021-02-09 06:44:59.999000+00:00"
  },
  "gate_summary": {
    "passed": 1,
    "failed": 7,
    "total": 8,
    "overall_verdict": "NO-GO"
  },
  "sensitivity_min_ratio": 0.5746081806962058,
  "pure_trend_comparator": {
    "sharpe": 0.28842321693927386,
    "trades": 201,
    "negative_trade_fraction": 0.6268656716417911,
    "liquidation_timestamp": "2021-03-19 11:59:59.999000+00:00"
  },
  "authority_boundary": "No OL/REC/DEC/readiness/transition/ranking/scoring/paper-live/runtime/protected-path authority."
}


In [4]:
gate_results = [
  {
    "gate_id": "D2-006-G1: OOS Sharpe >= 1.0",
    "threshold": 1.0,
    "observed": -0.8318,
    "pass": false
  },
  {
    "gate_id": "D2-006-G2: DSR > 1.0 at 95% CI",
    "threshold": 1.0,
    "observed": "0.0; P(SR>1.0)=0.0000; SR benchmark=0.6640",
    "pass": false
  },
  {
    "gate_id": "D2-006-G3: Per-fold max DD <= 20%",
    "threshold": 0.2,
    "observed": 1.0,
    "pass": false
  },
  {
    "gate_id": "D2-006-G4: Parameter sensitivity min_ratio >= 0.70",
    "threshold": 0.7,
    "observed": "0.5746; computed at +/-25%",
    "pass": false
  },
  {
    "gate_id": "D2-006-G5: Cost drag <= 50% of gross PnL",
    "threshold": 0.5,
    "observed": "0.0341; gross_pnl=-10353.03 <= 0: cost drag undefined/misleading",
    "pass": false
  },
  {
    "gate_id": "D2-006-G6: No single 90-day window > 60% of total PnL",
    "threshold": 0.6,
    "observed": 2.2493,
    "pass": false
  },
  {
    "gate_id": "D2-006-G7: <= 5 free parameters",
    "threshold": 5.0,
    "observed": 3.0,
    "pass": true
  },
  {
    "gate_id": "D2-006-G8: Walk-forward efficiency > 0.5",
    "threshold": 0.5,
    "observed": 0.0,
    "pass": false
  }
]
print(json.dumps(gate_results, indent=2))


[
  {
    "gate_id": "D2-006-G1: OOS Sharpe >= 1.0",
    "threshold": 1.0,
    "observed": -0.8318,
    "pass": false
  },
  {
    "gate_id": "D2-006-G2: DSR > 1.0 at 95% CI",
    "threshold": 1.0,
    "observed": "0.0; P(SR>1.0)=0.0000; SR benchmark=0.6640",
    "pass": false
  },
  {
    "gate_id": "D2-006-G3: Per-fold max DD <= 20%",
    "threshold": 0.2,
    "observed": 1.0,
    "pass": false
  },
  {
    "gate_id": "D2-006-G4: Parameter sensitivity min_ratio >= 0.70",
    "threshold": 0.7,
    "observed": "0.5746; computed at +/-25%",
    "pass": false
  },
  {
    "gate_id": "D2-006-G5: Cost drag <= 50% of gross PnL",
    "threshold": 0.5,
    "observed": "0.0341; gross_pnl=-10353.03 <= 0: cost drag undefined/misleading",
    "pass": false
  },
  {
    "gate_id": "D2-006-G6: No single 90-day window > 60% of total PnL",
    "threshold": 0.6,
    "observed": 2.2493,
    "pass": false
  },
  {
    "gate_id": "D2-006-G7: <= 5 free parameters",
    "threshold": 5.0,
    "observed": 3.

In [5]:
diagnostics = {
  "sensitivity": [
    {
      "variant": "trend_lookback_bars_up_25pct",
      "trend_lookback_bars": 120,
      "sharpe": -0.47793086311428856,
      "ratio_to_base": 0.5746081806962058,
      "trades": 128,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "trend_lookback_bars_down_25pct",
      "trend_lookback_bars": 72,
      "sharpe": -0.7298409150549051,
      "ratio_to_base": 0.8774753688528119,
      "trades": 216,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "funding_threshold_abs_up_25pct",
      "funding_threshold_abs": 0.000125,
      "sharpe": -1.2123829288852517,
      "ratio_to_base": 1.457627457943225,
      "trades": 157,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "funding_threshold_abs_down_25pct",
      "funding_threshold_abs": 7.5e-05,
      "sharpe": -2.080649780600004,
      "ratio_to_base": 2.5015299855423474,
      "trades": 58,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "min_hold_bars_up_25pct",
      "min_hold_bars": 5,
      "sharpe": -0.8317508855081367,
      "ratio_to_base": 1.0,
      "trades": 160,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "min_hold_bars_down_25pct",
      "min_hold_bars": 3,
      "sharpe": -0.8317508855081367,
      "ratio_to_base": 1.0,
      "trades": 160,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    }
  ],
  "pure_trend_comparator": {
    "sharpe": 0.28842321693927386,
    "max_drawdown": 1.0,
    "max_drawdown_peak": "2021-01-11 16:44:59.999000+00:00",
    "max_drawdown_trough": "2021-03-19 11:59:59.999000+00:00",
    "n_obs": 42575,
    "bars": 220128,
    "trades": 201,
    "hit_rate": 0.373134328358209,
    "profit_factor": 0.9482679919316774,
    "final_equity": 0.0,
    "total_pnl": -10000.0,
    "realized_pnl": -2029.8726688193678,
    "unrealized_pnl": 0.0,
    "fees_paid": 2883.777903999999,
    "slippage_paid": 1441.8889519999996,
    "funding_paid": 3644.460475180702,
    "gross_pnl": -2029.8726688193678,
    "total_cost": 7970.127331180701,
    "cost_drag_fraction": -3.926417382532844,
    "skipped_bars": 0,
    "liquidated": true,
    "liquidation_timestamp": "2021-03-19 11:59:59.999000+00:00",
    "negative_trade_fraction": 0.6268656716417911,
    "side_summary": [
      {
        "direction": -1,
        "count": 101,
        "net_pnl_sum": -27939.503334819354,
        "net_pnl_mean": -276.62874588930055
      },
      {
        "direction": 1,
        "count": 100,
        "net_pnl_sum": 23748.947289999986,
        "net_pnl_mean": 237.48947289999987
      }
    ],
    "signal_counts": {
      "-1": 103019,
      "0": 104,
      "1": 117005
    },
    "actual_bar_window_start": "2020-01-01 00:14:59.999000+00:00",
    "actual_bar_window_end": "2026-04-11 23:59:59.999000+00:00"
  }
}
print(json.dumps(diagnostics, indent=2))


{
  "sensitivity": [
    {
      "variant": "trend_lookback_bars_up_25pct",
      "trend_lookback_bars": 120,
      "sharpe": -0.47793086311428856,
      "ratio_to_base": 0.5746081806962058,
      "trades": 128,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "trend_lookback_bars_down_25pct",
      "trend_lookback_bars": 72,
      "sharpe": -0.7298409150549051,
      "ratio_to_base": 0.8774753688528119,
      "trades": 216,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "funding_threshold_abs_up_25pct",
      "funding_threshold_abs": 0.000125,
      "sharpe": -1.2123829288852517,
      "ratio_to_base": 1.457627457943225,
      "trades": 157,
      "max_drawdown": 1.0,
      "final_equity": 0.0,
      "liquidated": true
    },
    {
      "variant": "funding_threshold_abs_down_25pct",
      "funding_threshold_abs": 7.5e-05,
      "sharpe": -2.080649780600004,
      